# LCDrive Val Per-Category Model Comparison

Loads the per-category evaluation tables produced by `scripts/lcdrive_per_category.py`
for multiple models/runs on the same LCDrive val clip set, and combines them into a
single side-by-side comparison table: one row per scenario category, with the
clip count plus each model's `min_ade` and `ade`.

## 1. Import Required Libraries

In [3]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

## 2. Define/Load Input Data

Each entry below is a `(model_name, csv_path)` pair pointing at a per-category
table produced by `scripts/lcdrive_per_category.py` (columns: `category`,
`n_clips`, `min_ade`, ..., `ade`, ...). All tables here were computed on the
same stratified 1,000-clip LCDrive val subset
(`lcdrive_val_primary_scenario_mysubset.csv`), so clip counts per category
should match across models.

In [4]:
# (model_name, csv_path) pairs -- edit/add entries here to compare more runs.
MODEL_TABLES = {
    "Stage1_2B": (
        "/data/01/achahe/alpamayo-recipes/recipes/alpamayo1_5_sft/training/"
        "output_stage1_cosmos2b_lcdrive_4gpu_10ep/lcdrive_val_by_category_mysubset.csv"
    ),
    "Token_10B": (
        "/data/01/achahe/alpamayo-recipes/recipes/alpamayo1_5_sft/training/"
        "output_eval_10b_token_lcdrive_mysubset/lcdrive_val_by_category_mysubset.csv"
    ),
    "32B": (
        "/home/achahe/alpamayo2/outputs/"
        "per_category_ade_table.csv"
    )
}

model_dfs = {}
for model_name, csv_path in MODEL_TABLES.items():
    path = Path(csv_path)
    if not path.exists():
        print(f"[warn] missing table for {model_name}: {path}")
        continue
    df = pd.read_csv(path)
    model_dfs[model_name] = df
    print(f"Loaded {model_name}: {len(df)} rows from {path}")

model_dfs[next(iter(model_dfs))].head()

Loaded Stage1_2B: 16 rows from /data/01/achahe/alpamayo-recipes/recipes/alpamayo1_5_sft/training/output_stage1_cosmos2b_lcdrive_4gpu_10ep/lcdrive_val_by_category_mysubset.csv
Loaded Token_10B: 16 rows from /data/01/achahe/alpamayo-recipes/recipes/alpamayo1_5_sft/training/output_eval_10b_token_lcdrive_mysubset/lcdrive_val_by_category_mysubset.csv
Loaded 32B: 16 rows from /home/achahe/alpamayo2/outputs/per_category_ade_table.csv


,category,n_clips,min_ade,min_ade/by_t=0.5,min_ade/by_t=1.0,min_ade/by_t=3.0,min_ade/by_t=5.0,ade,ade/by_t=3.0,corner_distance
0,Turning Maneuver,20,2.907099,0.019954,0.069063,0.609533,1.761185,5.481260,1.076203,2.838415
1,Lane Keeping Curve,59,2.833501,0.023360,0.075415,0.662882,1.779723,4.875821,0.828393,2.827036
2,Lane Change,47,1.980856,0.021702,0.064541,0.455141,1.203093,2.907930,0.642109,1.962862
3,Speed Control,53,1.757114,0.018468,0.058597,0.432156,1.091842,2.772551,0.578633,1.753235
4,Merging,44,1.699992,0.014812,0.051765,0.439278,1.107402,2.610608,0.527235,1.677485


## 3. Combine Data into Single DataFrame

Merge every model's table on `category`, keeping a single `n_clips` column
(clip counts should agree across models since they all use the same subset --
any mismatch is flagged), plus one `min_ade` and `ade` column pair per model.

In [5]:
combined = None

for model_name, df in model_dfs.items():
    sub = df[["category", "n_clips", "min_ade", "ade"]].copy()
    sub = sub.rename(
        columns={
            "n_clips": f"n_clips_{model_name}",
            "min_ade": f"{model_name}_minADE",
            "ade": f"{model_name}_ADE",
        }
    )
    combined = sub if combined is None else combined.merge(sub, on="category", how="outer")

# Sanity-check that clip counts agree across models per category, then
# collapse into a single n_clips column.
n_clips_cols = [c for c in combined.columns if c.startswith("n_clips_")]
n_clips_check = combined[n_clips_cols]
mismatched = combined.loc[n_clips_check.nunique(axis=1) > 1, ["category", *n_clips_cols]]
if not mismatched.empty:
    print("[warn] n_clips mismatch across models for these categories:")
    display(mismatched)

combined["n_clips"] = n_clips_check.bfill(axis=1).iloc[:, 0]
combined = combined.drop(columns=n_clips_cols)

combined.head()

,category,Stage1_2B_minADE,Stage1_2B_ADE,Token_10B_minADE,Token_10B_ADE,32B_minADE,32B_ADE,n_clips
0,ALL,1.232226,2.250203,0.858434,1.608950,0.711311,1.635660,1000
1,Cut-In,1.035082,2.057452,0.952929,1.518457,0.764280,1.902763,36
2,General Driving,0.814351,1.517976,0.658566,1.273175,0.525387,1.254211,341
3,Intersection Navigation,1.112061,2.045252,0.804136,1.428342,0.752473,1.991205,42
4,Lane Change,1.980856,2.907930,1.309720,2.182655,1.251496,2.283878,47


## 4. Format and Display Final Table

Order columns as `category, n_clips, <Model1>_minADE, <Model1>_ADE, <Model2>_minADE, <Model2>_ADE, ...`,
put the overall `ALL` row last, sort the rest by category name, and display
with light styling (bold `ALL` row, gradient highlighting on ADE columns).

In [7]:
metric_cols = []
for model_name in model_dfs:
    metric_cols.append(f"{model_name}_minADE")
    # metric_cols.append(f"{model_name}_ADE")

final_cols = ["category", "n_clips", *metric_cols]
final = combined[final_cols].copy()

# Put "ALL" (overall row) last, sort everything else alphabetically by category.
is_all = final["category"].eq("ALL")
final = pd.concat(
    [
        final[~is_all].sort_values("category").reset_index(drop=True),
        final[is_all].reset_index(drop=True),
    ],
    ignore_index=True,
)
final["n_clips"] = final["n_clips"].astype(int)

def _bold_all_row(row: pd.Series) -> list[str]:
    return ["font-weight: bold" if row["category"] == "ALL" else "" for _ in row]

styled = (
    final.style.hide(axis="index")
    .format({c: "{:.4f}" for c in metric_cols})
    .apply(_bold_all_row, axis=1)
    .background_gradient(subset=metric_cols, cmap="RdYlGn_r", axis=1)
)

display(styled)


category,n_clips,Stage1_2B_minADE,Token_10B_minADE,32B_minADE
Cut-In,36,1.0351,0.9529,0.7643
General Driving,341,0.8144,0.6586,0.5254
Intersection Navigation,42,1.1121,0.8041,0.7525
Lane Change,47,1.9809,1.3097,1.2515
Lane Keeping,53,1.3351,0.8910,0.8447
Lane Keeping Curve,59,2.8335,1.4043,1.0939
Lead Vehicle Following,56,0.9715,0.8995,0.6532
Merging,44,1.7000,0.9787,1.3279
Nudge Maneuver,53,1.2198,0.8090,0.6898
Nudge Static Obstacle Maneuver,56,1.0746,0.6861,0.7049


## 5. Export Combined Table (Optional)

Write the combined comparison table out to CSV for reporting/sharing.

In [ ]:
out_path = Path("/home/achahe/alpamayo-recipes/lcdrive_physicalai_av_manifests/lcdrive_val_model_comparison_mysubset.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
final.to_csv(out_path, index=False)
print(f"Wrote combined comparison table to {out_path}")